#Setup & Config

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    current_timestamp,
    regexp_extract,
    col,
    lit
)
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType
)

CRICSHEET_SOURCE_PATH = "s3a://ipl-analytics-raw-data/cricsheet/extracted"
PLAYERS_SOURCE_PATH = "s3a://ipl-analytics-raw-data/scraped/raw_players.parquet"
MATCH_INFO_SOURCE_PATH = "s3://ipl-analytics-raw-data/cricsheet/raw_match_info.parquet"
BRONZE_DELIVERY_CHECKPOINT_PATH = "s3a://ipl-analytics-raw-data/checkpoints/bronze_deliveries/"
BRONZE_PLAYERS_CHECKPOINT_PATH = "s3a://ipl-analytics-raw-data/checkpoints/bronze_players/"
SCHEMA_LOCATION_PATH = "s3a://ipl-analytics-raw-data/checkpoints/schema/"

# Bronze table
BRONZE_RAW_DELIVERY_TABLE = "ipl_catalog.bronze.raw_deliveries"
BRONZE_RAW_MATCH_INFO_TABLE = "ipl_catalog.bronze.raw_match_info"
BRONZE_RAW_PLAYER_TABLE = "ipl_catalog.bronze.raw_players"

# Create catalogs and schema

In [0]:
# create catalog
spark.sql("CREATE CATALOG IF NOT EXISTS ipl_catalog")

# Create raw_data, BRONZE, SILVER and GOLD schemas (DATABASE)
spark.sql("CREATE SCHEMA IF NOT EXISTS ipl_catalog.raw_data")
spark.sql("CREATE SCHEMA IF NOT EXISTS ipl_catalog.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS ipl_catalog.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS ipl_catalog.gold")

# Create a volume - this is like a folder in Unity Catalog
spark.sql("CREATE VOLUME IF NOT EXISTS ipl_catalog.raw_data.cricsheet_files")
spark.sql("CREATE VOLUME IF NOT EXISTS ipl_catalog.raw_data.checkpoints")
spark.sql("CREATE VOLUME IF NOT EXISTS ipl_catalog.raw_data.bronze")

# Create tables inside BRONZE schema
spark.sql(f"CREATE TABLE IF NOT EXISTS {BRONZE_RAW_DELIVERY_TABLE}")
spark.sql(f"CREATE TABLE IF NOT EXISTS {BRONZE_RAW_MATCH_INFO_TABLE}")
spark.sql(f"CREATE TABLE IF NOT EXISTS {BRONZE_RAW_PLAYER_TABLE}")

# Create directories for checkpoint
dbutils.fs.mkdirs(BRONZE_DELIVERY_CHECKPOINT_PATH)
dbutils.fs.mkdirs(BRONZE_PLAYERS_CHECKPOINT_PATH)
dbutils.fs.mkdirs(SCHEMA_LOCATION_PATH)

# Bronze Raw Delivery
Delivery Schema Definition

In [0]:
"""
Always define schema explicitely with AutoLoader -never let it infer on streaming read
"""

# Cricsheet CSV2 delivery format columns
delivery_schema = StructType([
    StructField('match_id',     StringType(), True),
    StructField('season',       StringType(), True),
    StructField('start_date',   StringType(), True),
    StructField('venue',        StringType(), True),
    StructField('innings',      StringType(), True),
    StructField('ball',         StringType(), True),
    StructField('batting_team', StringType(), True),
    StructField('bowling_team', StringType(), True),
    StructField('striker',      StringType(), True),
    StructField('non_striker',  StringType(), True),
    StructField('bowler',       StringType(), True),
    StructField('runs_off_bat', StringType(), True),
    StructField('extras',       StringType(), True),
    StructField('wides',        StringType(), True),
    StructField('noballs',      StringType(), True),
    StructField('byes',         StringType(), True),
    StructField('legbyes',      StringType(), True),
    StructField('penalty',      StringType(), True),
    StructField('wicket_type',  StringType(), True),
    StructField('player_dismissed', StringType(), True),
    StructField('other_wicket_type', StringType(), True),
    StructField('other_player_dismissed', StringType(), True),
])

# all the columns are StringType as Bronze - cast to correct types in Silver
# this ensures no data is ever rejected at ingestion
print(f"Schema defined with {len(delivery_schema.fields)} columns")

Auto Loader stream

In [0]:
def load_deliveries_to_bronze() :

    # read stream using Auto Loader
    df_raw = (
        spark.readStream
        .format("cloudFiles") # Auto Loader format
        .option("cloudFiles.format", "csv") # Source file format
        .option("cloudFiles.schemaLocation", SCHEMA_LOCATION_PATH) # Schema tracking
        .option("header", "true") # csvs have header
        .option("rescuedDataColumn", "_rescued_data") # capture unparsed data
        .schema(delivery_schema)
        .load(CRICSHEET_SOURCE_PATH)
    )

    # add metadata columns
    df_enriched = (
        df_raw
        .withColumn("_source_file", col("_metadata.file_path"))
        .withColumn("_file_size", col("_metadata.file_size"))
        .withColumn("_file_modified", col("_metadata.file_modification_time"))
        .withColumn("_ingested_at", current_timestamp())
        .withColumn("_source", lit("cricsheet"))
        # extract match_id from filename: "1234567.csv" -> "1234567"
        .withColumn("_file_match_id", regexp_extract(col("_metadata.file_path"), "(\d+).csv$", 1))
        # fix season format: 2020/21 -> "2020"
        .withColumn("season", col("season").substr(1, 4))
    )

    # write stream to Delta table
    query = (
        df_enriched
        .writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", BRONZE_DELIVERY_CHECKPOINT_PATH)
        .option("mergeSchema", "true") # handles new columns gracefully
        .trigger(availableNow = True) # process all files then stop- preferred for Batch processing
        .toTable(BRONZE_RAW_DELIVERY_TABLE)
    )

    query.awaitTermination()
    print("Bronze delivery load complete")

load_deliveries_to_bronze()

Verify the load

In [0]:
df_bronze_raw_deliveries = spark.table(BRONZE_RAW_DELIVERY_TABLE)
print(f"Total rows: {df_bronze_raw_deliveries.count():,}")

# check metadata columns
df_bronze_raw_deliveries.select(
    "_source_file",
    "_ingested_at",
    "_source",
    "season",
    "batting_team",
    "runs_off_bat"
).show(5, truncate = False)

# season loaded - should be 2008 to 2024
df_bronze_raw_deliveries.groupBy("season").count().orderBy("season").show(20)

# check for any rescued (unparseable) rows
rescued = df_bronze_raw_deliveries.filter(col("_rescued_data").isNotNull())
print(f"Rescued rows (parse failures): {rescued.count()}")

Optimize the Raw delivery-Bronze table

In [0]:
# run after initial load - improves query speed significantly
spark.sql(f"OPTIMIZE {BRONZE_RAW_DELIVERY_TABLE}")

# check table details
spark.sql(f"DESCRIBE DETAIL {BRONZE_RAW_DELIVERY_TABLE}").show(vertical = True)

# see table history - this is Delta lake time travel
spark.sql(f"DESCRIBE HISTORY {BRONZE_RAW_DELIVERY_TABLE}").show(5)

# Bronze Match info
Read teh scraped players parquet file

In [0]:
# Read the parquet file scraped locally
df_info_raw = spark.read.parquet(MATCH_INFO_SOURCE_PATH)
print(f"Raw info rows loaded from scraped parquet: {df_info_raw.count()}")
df_info_raw.show(10, truncate = False)

Inspect the raw structure

In [0]:
print("Column names:", df_info_raw.columns)
print("\nDistinct keys found:")
print(f"Parsed rows: {df_info_raw.count():,}")
# df_info_raw.select("_c1").distinct().orderBy("_c1").show(50, truncate=False)

Parse and enrich with match ID and metadata

In [0]:
def parse_match_info(df_info_raw) :
  """
  Parses raw _info.csv content into a clean key-value DataFrame.
    
    Each row in the info CSV looks like:
    info, city, Mumbai
    info, players, Mumbai Indians, R Sharma   ← variable columns
    info, registry, people, R Sharma, uuid    ← 5 columns
    
    We handle variable columns by joining _c2 onwards as one value string.
  """
  from pyspark.sql.functions import (
    concat_ws,
    array,
    when
  )



Write Bronze Delta table

In [0]:
def write_bronze_match_info(df_parsed):
    spark.sql("CREATE SCHEMA IF NOT EXISTS ipl_catalog.bronze")
    
    (df_parsed
        .write
        .format("delta")
        .mode("overwrite")
        .option("mergeSchema", "true")
        .saveAsTable(BRONZE_RAW_MATCH_INFO_TABLE))
    
    count = spark.table(BRONZE_RAW_MATCH_INFO_TABLE).count()
    print(f"Bronze match info table created: {BRONZE_RAW_MATCH_INFO_TABLE}")
    print(f"Total rows: {count:,}")

write_bronze_match_info(df_info_raw)

Verify the Bronze table

In [0]:
df_check = spark.table(BRONZE_RAW_MATCH_INFO_TABLE)

# Total rows
print(f"Total rows: {df_check.count():,}")

# Unique matches loaded
unique_matches = df_check.select("match_id").distinct().count()
print(f"Unique matches: {unique_matches:,}")

# Check schema
df_check.printSchema()

# Sample rows
df_check.show(10, truncate=False)

# Verify no null match IDs
null_ids = df_check.filter(col("match_id").isNull()).count()
print(f"Null match IDs: {null_ids} {'✓' if null_ids == 0 else '✗ FIX NEEDED'}")

# Bronze Players
Read the scraped players parquet file

In [0]:
# Read the parquet file scraped locally
df_players_raw = spark.read.parquet(PLAYERS_SOURCE_PATH)
print(f"Rows read from scraped parquet: {df_players_raw.count()}")
df_players_raw.show(5, truncate = False)

Add Bronze metadata column

In [0]:
df_players_bronze = (
    df_players_raw
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source", lit("cricinfo_squads"))
    # season already exists from scarper - just clean it
    .withColumn("season", col("season").cast(StringType()))
)

df_players_bronze.printSchema()

Write to Bronze deltatable

In [0]:
(df_players_bronze
 .write
 .format("delta")
 .mode("overwrite")
 .option("mergeSchema", "true")
 .saveAsTable(BRONZE_RAW_PLAYER_TABLE)
)

print(f"Bronze players table created: {BRONZE_RAW_PLAYER_TABLE}")
print(f"Row count: {spark.table(BRONZE_RAW_PLAYER_TABLE).count()}")

Verify row count and team breakdown

In [0]:
df_check = spark.table(BRONZE_RAW_PLAYER_TABLE)

# check all teams are present - should be 10 IPL teams
print("Teams scraped:")
df_check.groupBy("team").count().orderBy("team").show(truncate = False)

# check columns
df_check.show(5, truncate = False)

# Test Work

In [0]:
# Run this after load_deliveries_to_bronze()
# Check if table exists at all
print(spark.catalog.tableExists("ipl_catalog.bronze.raw_deliveries"))

# Check row count
print(spark.table("ipl_catalog.bronze.raw_deliveries").count())

# Check table history — did any write happen?
spark.sql("DESCRIBE HISTORY ipl_catalog.bronze.raw_deliveries").show(5, truncate=False)

In [0]:
# Verify files are visible at your source path
files = dbutils.fs.ls("s3a://ipl-analytics-raw-data/cricsheet/extracted/")
print(f"Files visible to Databricks: {len(files)}")
print(f"Sample files:")
for f in files[:5]:
    print(f"  {f.path}  size={f.size}")

In [0]:
# List checkpoint contents
checkpoint_path = "s3a://ipl-analytics-raw-data/checkpoints/bronze_deliveries/"

try:
    checkpoint_files = dbutils.fs.ls(checkpoint_path)
    print(f"Checkpoint files: {len(checkpoint_files)}")
    for f in checkpoint_files:
        print(f"  {f.path}")
except Exception as e:
    print(f"Checkpoint path empty or missing: {e}")

In [0]:
# Simple batch read — no streaming, no checkpoint
# If this returns rows, the files are readable
df_test = (
    spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "false")
        .load("s3a://ipl-analytics-raw-data/cricsheet/extracted/")
)

print(f"Batch read row count: {df_test.count():,}")
df_test.show(5)

In [0]:
from pyspark.sql.functions import current_timestamp, lit, col, regexp_extract
from pyspark.sql.types import *

# ── Paths ──────────────────────────────────────────────────────
CRICSHEET_SOURCE_PATH = "s3a://ipl-analytics-raw-data/cricsheet/extracted/"
CHECKPOINT_PATH       = "s3a://ipl-analytics-raw-data/checkpoints/bronze_deliveries/"
SCHEMA_PATH           = "s3a://ipl-analytics-raw-data/checkpoints/schema/"

# ── Clean checkpoint first ─────────────────────────────────────
# If checkpoint exists from a previous failed run,
# Auto Loader thinks all files are already processed → writes 0 rows
# Delete it and start fresh
try:
    dbutils.fs.rm(CHECKPOINT_PATH, recurse=True)
    dbutils.fs.rm(SCHEMA_PATH, recurse=True)
    print("Checkpoint cleared")
except:
    print("No checkpoint to clear")

# Recreate checkpoint folders
dbutils.fs.mkdirs(CHECKPOINT_PATH)
dbutils.fs.mkdirs(SCHEMA_PATH)
print("Checkpoint folders recreated")